In [5]:
import glob

import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
print(os.getcwd())

/Users/dplavos/Desktop/PERCEPTION/perception-for-autonomous-systems/week_4


In [6]:
# img = cv2.imread(images_left[0])

# plt.figure(figsize=(12, 8))
# plt.imshow(img[..., [2, 1, 0]])
# plt.show()

In [7]:
nb_vertical = 6
nb_horizontal = 8

square_size = 33.6  # Millimetres.

# Keep the same coordinate construction as your original exercise.
# OpenCV interprets the pattern tuple as (columns, rows).
# Therefore, this setup searches for 6 corners per row and 7 rows.
objp = np.zeros((nb_horizontal * nb_vertical, 3), np.float32)
objp[:, :2] = np.mgrid[0:nb_vertical, 0:nb_horizontal].T.reshape(-1, 2)
objp = objp * square_size

# Arrays to store object points and image points from both cameras.
objpoints = []
imgpoints_left = []
imgpoints_right = []

# Keep track of image pairs where both detections succeed.
good_images_left = []
good_images_right = []

images_left = sorted(glob.glob("rs/left-*.png"))
images_right = sorted(glob.glob("rs/right-*.png"))

assert images_left, "No left images found. Check the path."
assert images_right, "No right images found. Check the path."
assert len(images_left) == len(images_right), "Different image counts."

# Optimization parameters for refining corner positions.
criteria = (
    cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
    30,
    0.001,
)

In [8]:
image_size = None

try:
    for fname_left, fname_right in zip(images_left, images_right):
        img_left = cv2.imread(fname_left)
        img_right = cv2.imread(fname_right)

        assert img_left is not None, fname_left
        assert img_right is not None, fname_right

        h, w = img_left.shape[:2]

        assert img_right.shape[:2] == (h, w), "Different image sizes."

        if image_size is None:
            image_size = (w, h)

        assert image_size == (w, h), "Calibration images must share a resolution."

        gray_left = cv2.cvtColor(img_left, cv2.COLOR_BGR2GRAY)
        gray_right = cv2.cvtColor(img_right, cv2.COLOR_BGR2GRAY)

        ret_left, corners_left = cv2.findChessboardCorners(
            gray_left, (nb_vertical, nb_horizontal), None
        )

        ret_right, corners_right = cv2.findChessboardCorners(
            gray_right, (nb_vertical, nb_horizontal), None
        )

        # Only use pairs where the board is detected in both images.
        if ret_left == True and ret_right == True:
            corners_left = cv2.cornerSubPix(
                gray_left, corners_left, (11, 11), (-1, -1), criteria
            )

            corners_right = cv2.cornerSubPix(
                gray_right, corners_right, (11, 11), (-1, -1), criteria
            )

            objpoints.append(objp)

            imgpoints_left.append(corners_left)
            imgpoints_right.append(corners_right)

            good_images_left.append(fname_left)
            good_images_right.append(fname_right)

            # Draw and display the corners.
            img_left = cv2.drawChessboardCorners(
                img_left, (nb_vertical, nb_horizontal), corners_left, ret_left
            )

            img_right = cv2.drawChessboardCorners(
                img_right, (nb_vertical, nb_horizontal), corners_right, ret_right
            )

            cv2.imshow("Left image", img_left)
            cv2.imshow("Right image", img_right)
            cv2.waitKey(50)

finally:
    cv2.destroyAllWindows()

    # Let macOS process the window-closing events.
    for _ in range(10):
        cv2.waitKey(1)

print("Number of successful image pairs:")
print(len(objpoints))

assert len(objpoints) >= 3, "Too few successful pairs for calibration."

Number of successful image pairs:
8
